# Semantic Autoscaling: Test Results

This Google Colab notebook presents the verified simulation results for the semantic-archetype-aware autoscaling prototype.

**Claim under test:** at approximately constant aggregate request volume, conditioning capacity and cost-governance decisions on workload archetype reduces latency-SLA violations and improves budget-aware behavior compared with the aggregate-only baseline.

The notebook reads committed CSV and JSON artifacts. It does not call AWS and does not fabricate new measurements.

## Colab setup

Upload the repository ZIP to Colab, or clone the GitHub repository. Set `REPO` to the extracted repository path if needed.

In [ ]:
# Option A: clone the repository
!git clone -q https://github.com/Rohithpranov07/Cloud.git

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

REPO = Path('/content/Cloud')
DATA = REPO / 'data'
assert DATA.exists(), f'Could not find data directory: {DATA}'
sns.set_theme(style='whitegrid', palette='colorblind')
pd.set_option('display.max_columns', None)
print(f'Repository: {REPO}')
print(f'Artifacts: {len(list(DATA.iterdir()))} files')

In [ ]:
evaluation = pd.read_csv(DATA / 'evaluation_results.csv')
+    multiseed = pd.read_csv(DATA / 'multiseed_results.csv')
+    sensitivity = pd.read_csv(DATA / 'sensitivity.csv')
+    latency = json.loads((DATA / 'latency_summary.json').read_text())
+
+    print('Evaluation rows:', len(evaluation))
+    print('Multiseed rows:', len(multiseed))
+    print('Sensitivity rows:', len(sensitivity))
+    display(evaluation.head())

## 1. Headline result

The paired multi-seed evaluation uses 30 seeds. The sizing rule is reported explicitly because `round` is the frozen default and `ceil` is the sensitivity option.

In [ ]:
def headline(sizing):
    result = latency[sizing]
    rows = []
    for metric, values in result.items():
        rows.append({
            'sizing': sizing,
            'metric': metric,
            'n': values['n'],
            'baseline_median': values['median_baseline'],
            'aware_median': values['median_aware'],
            'paired_difference': values['median_paired_diff'],
            'p_value': values['p_value'],
            'rank_biserial_r': values['rank_biserial_r'],
        })
+    return pd.DataFrame(rows)
+
+headline_table = pd.concat([headline('round'), headline('ceil')], ignore_index=True)
+display(headline_table)
+
+plot_data = headline_table[headline_table['metric'].isin(['sla_violation_rate', 'p95_response_s'])].copy()
+plot_data['metric'] = plot_data['metric'].map({'sla_violation_rate': 'SLA violation rate', 'p95_response_s': 'p95 response (s)'})
+plot_data = plot_data.melt(id_vars=['sizing', 'metric'], value_vars=['baseline_median', 'aware_median'], var_name='policy', value_name='value')
+plot_data['policy'] = plot_data['policy'].map({'baseline_median': 'Baseline', 'aware_median': 'Archetype-aware'})
+g = sns.catplot(data=plot_data, x='sizing', y='value', hue='policy', col='metric', kind='bar', height=4, aspect=1.1)
+g.set_axis_labels('Sizing rule', 'Median value')
+g.set_titles('{col_name}')
+plt.show()

### Interpretation

- The baseline is aggregate-rate-only and cannot distinguish expensive agentic work from short conversational work.
- The aware controller forecasts each archetype independently and allocates capacity to the primitive that needs it.
- The p-values and effect sizes are reported measurements, not correctness gates. Re-run the repository tests separately for acceptance.

## 2. Per-seed latency and SLA comparison

In [ ]:
round_results = multiseed[multiseed['sizing'].eq('round')]
+    ceil_results = multiseed[multiseed['sizing'].eq('ceil')]
+
+    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
+    for frame, label, color in [(round_results, 'round', '#fc5000'), (ceil_results, 'ceil', '#524ae9')]:
+        axes[0].scatter(frame['baseline_p95_response_s'], frame['aware_p95_response_s'], label=label, alpha=.7, color=color)
+    max_value = axes[0].get_xlim()[1]
+    axes[0].plot([0, max_value], [0, max_value], '--', color='#070607', linewidth=1)
+    axes[0].set_title('p95 response: aware vs baseline')
+    axes[0].set_xlabel('Baseline p95 response (s)')
+    axes[0].set_ylabel('Aware p95 response (s)')
+    axes[0].legend(title='Sizing')
+
+    sns.boxplot(data=multiseed.melt(id_vars=['seed', 'sizing'], value_vars=['baseline_sla_violation_rate', 'aware_sla_violation_rate'], var_name='policy', value_name='rate'), x='sizing', y='rate', hue='policy', ax=axes[1])
+    axes[1].set_title('SLA violation rate across seeds')
+    axes[1].set_xlabel('Sizing rule')
+    axes[1].set_ylabel('Violation rate')
+    axes[1].legend(title='Policy')
+    plt.tight_layout()
+    plt.show()
+
+display(multiseed.groupby('sizing')[['baseline_p95_response_s', 'aware_p95_response_s', 'baseline_sla_violation_rate', 'aware_sla_violation_rate']].median())

## 3. Constant-volume composition shift and controller response

This plot uses the evaluation trace to show the central mechanism: aggregate volume remains nearly flat while the agentic component changes.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
+    axes[0].plot(evaluation['minute'], evaluation['baseline_capacity'], label='Baseline capacity', linewidth=2, color='#524ae9')
+    axes[0].plot(evaluation['minute'], evaluation['aware_capacity_total'], label='Aware total capacity', linewidth=2, color='#fc5000')
+    axes[0].axvspan(60, 75, color='#f5f28e', alpha=.7, label='Composition shift')
+    axes[0].set_ylabel('Capacity units')
+    axes[0].set_title('Capacity response during the shift')
+    axes[0].legend()
+
+    axes[1].plot(evaluation['minute'], evaluation['baseline_projected_spend'], label='Baseline projected spend', color='#524ae9')
+    axes[1].plot(evaluation['minute'], evaluation['aware_projected_spend'], label='Aware projected spend', color='#fc5000')
+    axes[1].axhline(24.0, linestyle='--', color='#070607', label='Simulation budget')
+    axes[1].axvspan(60, 75, color='#f5f28e', alpha=.7)
+    axes[1].set_xlabel('Minute')
+    axes[1].set_ylabel('Projected spend')
+    axes[1].set_title('Cost-governance projection')
+    axes[1].legend()
+    plt.tight_layout()
+    plt.show()
+
+display(evaluation.loc[evaluation['minute'].between(60, 75), ['minute', 'baseline_capacity', 'aware_capacity_total', 'baseline_projected_spend', 'aware_projected_spend']].head(16))

## 4. Sensitivity analysis

The sensitivity artifact varies agentic share and the service-time multiplier. Higher composition pressure should make the benefit of archetype-aware control more visible.

In [ ]:
pivot = sensitivity.pivot_table(index='agentic_share', columns='sla_multiplier', values='difference_baseline_minus_aware', aggfunc='median')
+    plt.figure(figsize=(10, 5))
+    sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlOrRd', center=0)
+    plt.title('Median SLA violation-rate improvement: baseline minus aware')
+    plt.xlabel('SLA multiplier')
+    plt.ylabel('Agentic share')
+    plt.tight_layout()
+    plt.show()
+    display(sensitivity.groupby(['agentic_share', 'sla_multiplier'])['difference_baseline_minus_aware'].median().reset_index())

## 5. Reproducibility and limitations

Run the repository acceptance suite separately from this presentation notebook:

```bash
pytest -q
mypy src/
```

Important limitations: serving pools are emulated, the live AWS layer is separate from the core simulation, the trace family is synthetic, SLA values were fixed before evaluation, and benchmark figures should be interpreted as measured project evidence rather than production capacity guarantees.